# Experiment: Zhipu Theme Matching Reproduction

Objective:
- 复现长问题输入时 Zhipu Web Search 的超时与重试行为。
- 对比“整句问题”与“手工归一化主题词”在概念匹配和候选股生成中的差异。
- 判断日志里的 warning 来自哪个环节，以及它是否一定会导致最终请求失败。


## Setup And Reproducibility

- 这个 notebook 会优先从 `deploy/.env` 读取 Zhipu 相关配置，以便尽量贴近 Docker 里的运行参数。
- 每次 probe 前会清空 intelligence / akshare adapter 的内存缓存，尽量暴露冷路径。
- 如需对比 HTTP 路径，请保证本地 `python-service` 已启动在 `127.0.0.1:8000`。


In [1]:
from __future__ import annotations

import io
import json
import logging
import os
import sys
import time
from contextlib import contextmanager
from pathlib import Path
from urllib.parse import quote

import requests


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"Unable to locate repo root from: {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PYTHON_SERVICES_ROOT = REPO_ROOT / "python_services"
if str(PYTHON_SERVICES_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SERVICES_ROOT))

QUESTION = "创新药出海链条里，哪些商业化指标最值得持续跟踪？"
NORMALIZED_THEME = "创新药出海"
LOCAL_BASE_URL = os.getenv("LOCAL_INTELLIGENCE_BASE_URL", "http://127.0.0.1:8000")

{
    "repo_root": str(REPO_ROOT),
    "python_services_root_exists": PYTHON_SERVICES_ROOT.exists(),
    "question": QUESTION,
    "normalized_theme": NORMALIZED_THEME,
    "local_base_url": LOCAL_BASE_URL,
}


{'repo_root': 'D:\\课外项目\\stock-screening-boost',
 'python_services_root_exists': True,
 'question': '创新药出海链条里，哪些商业化指标最值得持续跟踪？',
 'normalized_theme': '创新药出海',
 'local_base_url': 'http://127.0.0.1:8000'}

In [2]:
def load_env_from_deploy(names: set[str]) -> dict[str, str]:
    env_path = REPO_ROOT / "deploy" / ".env"
    loaded: dict[str, str] = {}
    if not env_path.exists():
        return loaded

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        if key not in names or os.getenv(key) is not None:
            continue
        os.environ[key] = value
        if key.endswith("API_KEY"):
            loaded[key] = "<set>" if value else "<empty>"
        else:
            loaded[key] = value
    return loaded


loaded_env = load_env_from_deploy(
    {
        "ZHIPU_API_KEY",
        "ZHIPU_WEB_SEARCH_MODEL",
        "ZHIPU_WEB_SEARCH_TIMEOUT_SECONDS",
        "ZHIPU_WEB_SEARCH_RETRIES",
    }
)
loaded_env


{'ZHIPU_API_KEY': '<set>',
 'ZHIPU_WEB_SEARCH_MODEL': 'glm-4-plus',
 'ZHIPU_WEB_SEARCH_TIMEOUT_SECONDS': '8',
 'ZHIPU_WEB_SEARCH_RETRIES': '2'}

In [3]:
from app.services import intelligence_data_adapter as ida_module
from app.services.akshare_adapter import AkShareAdapter
from app.services.intelligence_data_adapter import IntelligenceDataAdapter
from app.services.zhipu_search_client import ZhipuSearchClient

ida_module._ZHIPU_SEARCH_CLIENT = ZhipuSearchClient()


def reset_runtime_state() -> None:
    ida_module._CACHE.clear()
    AkShareAdapter.clear_caches()


def summarize_value(value):
    if isinstance(value, list):
        return {
            "type": "list",
            "count": len(value),
            "preview": value[:3],
        }
    if isinstance(value, dict):
        return value
    return repr(value)


@contextmanager
def capture_logs(*logger_names: str, level: int = logging.INFO):
    stream = io.StringIO()
    installed = []
    previous = []
    formatter = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
    for logger_name in logger_names:
        logger = logging.getLogger(logger_name)
        previous.append((logger, logger.level, logger.propagate))
        handler = logging.StreamHandler(stream)
        handler.setFormatter(formatter)
        handler.setLevel(level)
        logger.addHandler(handler)
        logger.setLevel(level)
        logger.propagate = False
        installed.append((logger, handler))
    try:
        yield stream
    finally:
        for logger, handler in installed:
            logger.removeHandler(handler)
        for logger, old_level, old_propagate in previous:
            logger.setLevel(old_level)
            logger.propagate = old_propagate


def run_probe(label: str, fn, *args, reset: bool = True, **kwargs) -> dict:
    if reset:
        reset_runtime_state()
    started = time.perf_counter()
    with capture_logs(
        "app.services.zhipu_search_client",
        "app.services.intelligence_data_adapter",
        level=logging.INFO,
    ) as log_stream:
        try:
            value = fn(*args, **kwargs)
            error = None
        except Exception as exc:  # noqa: BLE001
            value = None
            error = f"{type(exc).__name__}: {exc}"
    elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
    return {
        "label": label,
        "elapsed_ms": elapsed_ms,
        "error": error,
        "value": summarize_value(value),
        "logs": log_stream.getvalue().strip(),
    }


def http_probe(path: str, *, timeout: int = 65) -> dict:
    started = time.perf_counter()
    try:
        response = requests.get(f"{LOCAL_BASE_URL}{path}", timeout=timeout)
        payload = response.json()
        error = None
    except Exception as exc:  # noqa: BLE001
        payload = None
        error = f"{type(exc).__name__}: {exc}"
        response = None
    return {
        "status": response.status_code if response is not None else None,
        "elapsed_ms": round((time.perf_counter() - started) * 1000, 2),
        "error": error,
        "payload": payload,
    }


runtime_config = {
    "zhipu_api_key_present": bool(ida_module._ZHIPU_SEARCH_CLIENT.api_key),
    "zhipu_model": ida_module._ZHIPU_SEARCH_CLIENT.model,
    "zhipu_timeout_seconds": ida_module._ZHIPU_SEARCH_CLIENT.timeout_seconds,
    "zhipu_retries": ida_module._ZHIPU_SEARCH_CLIENT.retries,
}
runtime_config


{'zhipu_api_key_present': True,
 'zhipu_model': 'glm-4-plus',
 'zhipu_timeout_seconds': 8.0,
 'zhipu_retries': 2}

## Plan

- Hypothesis: 长问题原样输入会让 Zhipu Web Search 更容易超时或返回弱匹配，手工归一化主题词会更稳定。
- Variables to sweep: `QUESTION` 与 `NORMALIZED_THEME`。
- Metrics to record: 总耗时、Zhipu 重试次数、概念匹配结果、候选股结果、是否出现 `No tables found`。


In [4]:
direct_zhipu_probes = [
    run_probe(
        label=f"direct_zhipu_search::{theme}",
        fn=ida_module._ZHIPU_SEARCH_CLIENT.search_theme_concepts,
        theme=theme,
        limit=5,
        reset=False,
    )
    for theme in [QUESTION, NORMALIZED_THEME]
]
direct_zhipu_probes


TypeError: run_probe() missing 2 required positional arguments: 'label' and 'fn'

In [ ]:
match_theme_probes = [
    run_probe(
        label=f"match_theme_concepts::{theme}",
        fn=IntelligenceDataAdapter.match_theme_concepts,
        theme=theme,
        limit=5,
    )
    for theme in [QUESTION, NORMALIZED_THEME]
]
match_theme_probes


In [ ]:
candidate_probes = [
    run_probe(
        label=f"get_candidates_strict::{theme}",
        fn=IntelligenceDataAdapter.get_candidates_strict,
        theme=theme,
        limit=6,
    )
    for theme in [QUESTION, NORMALIZED_THEME]
]
candidate_probes


In [ ]:
http_results = {
    "question_concepts": http_probe(
        f"/api/v1/intelligence/themes/{quote(QUESTION)}/concepts"
    ),
    "question_candidates": http_probe(
        f"/api/v1/market/themes/{quote(QUESTION)}/candidates"
    ),
    "normalized_concepts": http_probe(
        f"/api/v1/intelligence/themes/{quote(NORMALIZED_THEME)}/concepts"
    ),
    "normalized_candidates": http_probe(
        f"/api/v1/market/themes/{quote(NORMALIZED_THEME)}/candidates"
    ),
}
http_results


## Results

- 先看 `direct_zhipu_probes`：如果这里已经出现 3 次 timeout，说明慢点在 Zhipu 本身。
- 再看 `match_theme_probes`：这里能看到 Zhipu timeout 后是否成功回落到 `auto` / `whitelist`。
- 再看 `candidate_probes`：如果日志出现 `Failed to load concept constituents ... No tables found`，说明是 THS 概念成分股页面没有被解析出表格。
- 最后对比 `http_results`：确认服务对外返回的是成功、降级还是超时。


In [ ]:
summary = {
    "runtime_config": runtime_config,
    "direct_zhipu_labels": [item["label"] for item in direct_zhipu_probes],
    "match_theme_labels": [item["label"] for item in match_theme_probes],
    "candidate_labels": [item["label"] for item in candidate_probes],
    "http_labels": list(http_results),
}
summary


## Next Steps

- 如果长问题路径稳定超时，而归一化主题词路径明显更快，优先在行业研究入口先做“问题 -> 主题词”归一化。
- 如果 `No tables found` 主要集中在某些固定概念，给这些概念补人工规则或更保守的 fallback。
- 如果 Zhipu timeout 是主耗时来源，可以考虑降低重试次数、缩短单次 timeout，或对长问题默认跳过 Zhipu 搜索。
